# Data Preprocessing: Exercise

This exercise covers the cleaning and preprocessing of the **Boston Housing Dataset**, a dataset containing various information about housing in the Boston area.

The dataset contains the following variables:

| Column | Description |
|--------|-------------|
| `CRIM` | Per capita crime rate by town |
| `ZN` | Proportion of residential land zoned for lots over 25,000 sq.ft. |
| `INDUS` | Proportion of non-retail business acres per town |
| `CHAS` | Dummy variable indicating proximity to the Charles River |
| `NOX` | Nitric oxide concentration (parts per 10 million) |
| `RM` | Average number of rooms per dwelling |
| `AGE` | Proportion of owner-occupied units built prior to 1940 |
| `DIS` | Weighted distances to five Boston employment centres |
| `RAD` | Index of accessibility to radial highways |
| `TAX` | Full-value property tax rate per $10,000 |
| `PTRATIO` | Pupil-teacher ratio by town |
| `B` | 1000(Bk - 0.63)² where Bk is the proportion of Black residents by town |
| `LSTAT` | Percentage of lower-status population |
| `PRICE` | Median value of owner-occupied homes in $1,000s |

**Tasks:**

1. The number of rows and columns in the dataset is verified
2. The data type of each variable is checked
3. The number of missing values per column is reported
4. Columns with more than 30% missing values are dropped
5. Rows with more than 25% missing values are dropped
6. All rows where `PRICE` is missing are dropped
7. Remaining missing quantitative values are imputed with the column mean
8. Qualitative variables are encoded
9. Remaining missing qualitative values are imputed with the column mode
10. Features are standardized
11. The cleaned dataframe is saved as a TSV file named `housing_clean.tsv`

In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('dataset/housing_dirty.csv', index_col=0)

## Step 1 — Rows and columns

In [2]:
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head()

Rows: 506, Columns: 14


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,PRICE
0.0,LOW,18.0,2.31,NO,538.0,6575.0,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1.0,LOW,0.0,7.07,NO,469.0,6421.0,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2.0,LOW,0.0,7.07,NO,469.0,7185.0,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3.0,LOW,0.0,2.18,NO,458.0,6998.0,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4.0,LOW,0.0,2.18,NO,458.0,7147.0,54.2,6.0622,3.0,222.0,18.7,396.90,NaN,36.2


## Step 2 — Variable types

In [3]:
df.dtypes

CRIM        object
ZN         float64
INDUS      float64
CHAS        object
NOX        float64
RM         float64
AGE        float64
DIS        float64
RAD        float64
TAX        float64
PTRATIO    float64
B          float64
LSTAT      float64
PRICE      float64
dtype: object

## Step 3 — Missing values per column

In [4]:
missing = df.isnull().sum()
missing_pct = missing / len(df) * 100
pd.DataFrame({'missing': missing, 'missing_%': missing_pct.round(2)})

,missing,missing_%
CRIM,0,0.00
ZN,2,0.40
INDUS,3,0.59
CHAS,0,0.00
NOX,7,1.38
RM,5,0.99
AGE,4,0.79
DIS,5,0.99
RAD,3,0.59
TAX,2,0.40


## Step 4 — Drop columns with > 30% missing values

In [5]:
threshold_col = 0.30
cols_to_drop = missing_pct[missing_pct > threshold_col * 100].index.tolist()
print(f'Columns dropped (>{threshold_col*100:.0f}% missing): {cols_to_drop}')
df.drop(columns=cols_to_drop, inplace=True)
print(f'Shape after drop: {df.shape}')

Columns dropped (>30% missing): ['LSTAT']
Shape after drop: (506, 13)


## Step 5 — Drop rows with > 25% missing values

In [6]:
threshold_row = 0.25
row_missing_pct = df.isnull().sum(axis=1) / df.shape[1]
rows_to_drop = row_missing_pct[row_missing_pct > threshold_row].index
print(f'Rows dropped (>{threshold_row*100:.0f}% missing): {len(rows_to_drop)}')
df.drop(index=rows_to_drop, inplace=True)
print(f'Shape after drop: {df.shape}')

Rows dropped (>25% missing): 5
Shape after drop: (501, 13)


## Step 6 — Drop rows where PRICE is missing

In [7]:
before = len(df)
df.dropna(subset=['PRICE'], inplace=True)
print(f'Rows dropped (PRICE missing): {before - len(df)}')
print(f'Shape after drop: {df.shape}')

Rows dropped (PRICE missing): 4
Shape after drop: (497, 13)


## Step 7 — Mean imputation for quantitative missing values

In [9]:
num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    n = df[col].isnull().sum()
    if n > 0:
        df[col].fillna(df[col].mean(), inplace=True)
        print(f'  {col}: imputed {n} values with mean ({df[col].mean():.4f})')

print(f'\nRemaining missing (quantitative): {df[num_cols].isnull().sum().sum()}')


Remaining missing (quantitative): 0


## Step 8 — Encode qualitative variables

- `CRIM`: ordinal (LOW → 0, MODERATE → 1, HIGH → 2, VERY HIGH → 3)
- `CHAS`: binary (NO → 0, YES → 1)

In [10]:
crim_order = {'LOW': 0, 'MODERATE': 1, 'HIGH': 2, 'VERY HIGH': 3}
df['CRIM'] = df['CRIM'].map(crim_order)

df['CHAS'] = df['CHAS'].map({'NO': 0, 'YES': 1})

print('Encoded — CRIM value counts:')
print(df['CRIM'].value_counts().sort_index())
print('\nEncoded — CHAS value counts:')
print(df['CHAS'].value_counts().sort_index())

Encoded — CRIM value counts:
CRIM
0    127
1    120
2    126
3    124
Name: count, dtype: int64

Encoded — CHAS value counts:
CHAS
0    463
1     34
Name: count, dtype: int64


## Step 9 — Mode imputation for qualitative missing values

In [11]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    n = df[col].isnull().sum()
    if n > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  {col}: imputed {n} values with mode ({mode_val})')

print(f'Total remaining missing values: {df.isnull().sum().sum()}')

Total remaining missing values: 0


## Step 10 — Standardization

In [12]:
scaler = StandardScaler()
features = [c for c in df.columns if c != 'PRICE']
df[features] = scaler.fit_transform(df[features])
df.describe().round(3)

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,PRICE
count,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000,497.000
mean,0.000,0.000,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,-0.000,0.000,0.000,-0.000,22.494
std,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,1.001,9.188
min,-1.334,-0.491,-1.557,-0.271,-2.070,-2.798,-2.320,-0.300,-0.984,-1.313,-2.734,-3.888,5.000
25%,-1.334,-0.491,-0.877,-0.271,-0.158,0.060,-0.851,-0.299,-0.642,-0.770,-0.498,0.204,16.800
50%,0.448,-0.491,-0.213,-0.271,0.160,0.250,0.314,-0.298,-0.527,-0.469,0.294,0.380,21.200
75%,0.448,0.042,1.011,-0.271,0.634,0.457,0.909,-0.296,1.642,1.514,0.806,0.432,25.000
max,1.339,3.772,2.414,3.690,1.823,1.542,1.118,6.234,1.642,1.780,1.645,0.439,50.000


## Step 11 — Save cleaned dataset as TSV

In [13]:
output_path = 'dataset/housing_clean.tsv'
df.to_csv(output_path, sep='\t')
print(f'Saved to {output_path} — shape: {df.shape}')

Saved to dataset/housing_clean.tsv — shape: (497, 13)
